# Activity 2: Statistical Outlier Detection

**Week 6 Day 2 · Machine Learning for Data Cleaning**

## An outlier is not a fact about a row

In Activity 1 you dealt with values that were absent. Now you deal with values that are present, but wrong. Or unusual. Or perfectly correct and simply surprising.

That ambiguity is the whole difficulty. There is no test that tells you a value is an outlier, because **"outlier" is not a property a data point has**. It is a statement about how a point relates to a reference population, and the moment you change the reference, the answer changes.

A 250 pound adult is unremarkable among adults and extraordinary among children. A fare of 400 dollars is fraud in Manhattan and a normal airport run in some cities. Nothing about the number changed. The comparison did.

So the engineering question is never "is this an outlier". It is:

> **Compared to what, and what do we do about it?**

## Why start with statistics rather than ML

You could reach straight for an ML detector. You should not, for three reasons:

1. **The statistical methods are often sufficient.** A great many production quality checks are an IQR rule that took four lines to write.
2. **They are explainable.** When an analyst asks why a record was quarantined, "it was more than three standard deviations below the mean" is an answer. "The ensemble scored it -0.31" is not.
3. **They give you a baseline.** The same discipline as Activity 1 applies. If the ML detector cannot beat a Z-score, it has not earned its dependency.

In this activity you build the statistical toolkit and, more importantly, find precisely where it breaks. Activity 3 picks up from that breaking point.

## Learning objectives

By the end of this activity you will be able to:

1. Apply Tukey's IQR method, Z-scores, and Modified Z-scores, and state what each assumes.
2. Explain why the mean and standard deviation are the wrong tools for finding extreme values.
3. Demonstrate that univariate methods are blind to anomalies that only exist across columns.
4. Distinguish point, contextual, and collective anomalies.
5. Show, with labelled real world events, that statistical methods on a time series miss anomalies entirely.

---
## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_rows", 12)

This activity uses `matplotlib` for the plots and `scipy` for the chi-squared threshold in Part 5, on top of the base project's `pandas` and `numpy`. All four are declared in the repo-root `pyproject.toml`, so `uv sync` from the repository root installs them. Run that once if you have not since they were added.

The next cell builds `DATA_DIR` by walking up from wherever the notebook is running until it finds `pyproject.toml`, the marker for the repository root. That means the path resolves correctly whether you run this notebook from the course folder or from your own copy under `student-work/week6/day2/`, without you hardcoding it.

In [ ]:
from pathlib import Path


def find_repo_root(start=None):
    """Walk upward until we find the repo root (the folder holding pyproject.toml)."""
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repo root from " + str(start))


DATA_DIR = find_repo_root() / "Week 6" / "Labs" / "Day 2" / "data"

---
# Part 1: Look at the data first

Load a dataset of 10,000 human height and weight measurements.

In [ ]:
people = pd.read_csv(DATA_DIR / "weight-height.csv")
people.head()

In [ ]:
people.describe().round(2)

Nothing obviously broken. Heights run from 54 to 79 inches, weights from 65 to 270 pounds. All physically possible.

Before computing anything, plot it. Every statistical method you are about to use makes an assumption about the shape of this distribution, and you can check that assumption by eye in one chart.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ["Height", "Weight"]):
    ax.hist(people[col], bins=60, color="tab:blue", alpha=0.75)
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel(col)
plt.tight_layout()
plt.show()

Look closely at those shapes. Neither is a single clean bell curve. Both are wide and flat topped, and `Height` in particular hints at **two overlapping humps**.

There is a reason for that, and the dataset tells you what it is.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for gender, colour in [("Male", "tab:blue"), ("Female", "tab:orange")]:
    ax.hist(people.loc[people["Gender"] == gender, "Height"],
            bins=50, alpha=0.6, label=gender, color=colour)
ax.set_xlabel("Height (inches)")
ax.set_title("The population is a mixture of two distributions")
ax.legend()
plt.tight_layout()
plt.show()

This is a **mixture of two populations**, each roughly normal, with different means.

Remember this when the numbers start coming out strangely later. Every method in this notebook assumes it is looking at one population. It is not.

This is not a contrived teaching dataset quirk. Mixed populations are the normal case in production data. Retail transactions mix consumer and wholesale. Server latency mixes cache hits and cache misses. API traffic mixes humans and bots. If you compute one threshold across a mixture, you get a threshold that describes neither group.

---
# Part 2: Tukey's IQR method

The oldest and most robust approach. It uses quartiles rather than the mean.

- **Q1** is the value below which 25 percent of the data falls.
- **Q3** is the value below which 75 percent falls.
- **IQR** is `Q3 - Q1`, the range of the middle half of the data.

Anything further than `1.5 x IQR` beyond either quartile is flagged.

The reason this is robust is that quartiles are **positional**. Making the largest value in your dataset a million times larger does not move Q3 at all, because Q3 only cares about which value sits at that position.

In [ ]:
def iqr_outliers(series, k=1.5):
    """Flag values beyond k * IQR from the quartiles. Returns a boolean Series."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return (series < lower) | (series > upper)

In [ ]:
height_iqr = iqr_outliers(people["Height"])
weight_iqr = iqr_outliers(people["Weight"])

print("Height outliers by IQR:", height_iqr.sum())
print("Weight outliers by IQR:", weight_iqr.sum())

Out of 10,000 rows, a very small number. This distribution is well behaved at its edges.

A box plot draws exactly this rule, which is why box plots are the fastest visual outlier check available.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.boxplot([people["Height"], people["Weight"]], vert=False, tick_labels=["Height", "Weight"])
ax.set_title("Box plots draw the 1.5 x IQR rule directly")
plt.tight_layout()
plt.show()

## The multiplier is a policy decision

The 1.5 is a convention, not a law. It is a dial that trades false positives against missed detections.

In [ ]:
for k in [1.0, 1.5, 2.0, 3.0]:
    n = iqr_outliers(people["Height"], k=k).sum()
    print(f"k = {k:<4} flags {n:>4} rows ({n / len(people):.2%})")

`k` is not just a number in a printout. It is a boundary you can see move. Plot Height's distribution once, and draw the cutoff for each `k` from the loop above.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(people["Height"], bins=80, color="lightsteelblue", alpha=0.8)

q1, q3 = people["Height"].quantile([0.25, 0.75])
iqr = q3 - q1
for k, color in {1.0: "tab:green", 1.5: "tab:blue", 2.0: "tab:orange", 3.0: "tab:red"}.items():
    lower, upper = q1 - k * iqr, q3 + k * iqr
    ax.axvline(lower, color=color, ls="--", lw=1.5, label=f"k={k}")
    ax.axvline(upper, color=color, ls="--", lw=1.5)

ax.set_xlabel("Height (inches)")
ax.set_title("The IQR boundary moves with k, the histogram does not")
ax.legend()
plt.tight_layout()
plt.show()

Notice how much territory opens up between `k=1.0` and `k=3.0`. A `k` of 1.0 starts flagging values well inside the main body of the distribution, which is why it caught far more rows in the printout above. A `k` of 3.0 only catches the rare, genuine tail. There is no correct line in this picture. There is only the line that matches how much review capacity you have and how expensive a miss is.

Choosing `k` is a business conversation, not a statistical one. The right question is what each kind of mistake costs.

If a flagged record means a human reviews it, a high false positive rate is expensive in salary. If a missed record means a fraudulent claim gets paid, a missed detection is expensive in cash. Set `k` accordingly and document why.

---
# Part 3: Z-scores, and why they undermine themselves

The Z-score expresses each value as a number of standard deviations from the mean.

A common rule flags anything beyond 3.

In [ ]:
def z_score_outliers(series, threshold=3.0):
    z = (series - series.mean()) / series.std()
    return z.abs() > threshold

In [ ]:
print("Height outliers by Z-score:", z_score_outliers(people["Height"]).sum())
print("Weight outliers by Z-score:", z_score_outliers(people["Weight"]).sum())

Counts alone hide how these points sit relative to the threshold. Plot the Z-scores themselves, with the cutoff drawn in, and the flagged points marked.

In [ ]:
def plot_scores(series, threshold, label, ax):
    """Scatter a score series with its +/- threshold lines, and mark the flagged points."""
    ax.plot(series.index, series.values, "^", color="black", markersize=3, alpha=0.5)
    ax.axhline(threshold, color="tab:red", ls="--", lw=1.2)
    ax.axhline(-threshold, color="tab:red", ls="--", lw=1.2)
    flagged = series[series.abs() > threshold]
    ax.scatter(flagged.index, flagged.values, color="tab:red", s=30, zorder=3)
    ax.set_ylabel(label)


height_z = (people["Height"] - people["Height"].mean()) / people["Height"].std()

fig, ax = plt.subplots(figsize=(11, 3.5))
plot_scores(height_z, threshold=3.0, label="Z-score", ax=ax)
ax.set_xlabel("Row index")
ax.set_title("Height Z-scores, threshold at +/-3.0")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
plot_scores(height_z, threshold=2.0, label="Z (t=2.0)", ax=axes[0])
plot_scores(height_z, threshold=3.0, label="Z (t=3.0)", ax=axes[1])
axes[0].set_title("Same scores, two thresholds: the line moves, the points do not")
axes[-1].set_xlabel("Row index")
plt.tight_layout()
plt.show()

Same Z-scores in both panels. Only the dashed line moved. At `t=2.0` more points fall outside it and get flagged; at `t=3.0` the line sits further out and only the most extreme survive. This is the same policy dial as `k` in the IQR method, just measured in standard deviations instead of quartile widths.

Broadly similar to IQR here. But the Z-score has a structural weakness that this clean dataset hides, and it is worth seeing plainly.

**The mean and standard deviation are themselves distorted by the outliers you are hunting.**

Take a small clean sample and inject one extreme value.

In [ ]:
clean = pd.Series([10, 11, 12, 11, 10, 12, 11, 10, 11, 12], dtype=float)
contaminated = pd.concat([clean, pd.Series([500.0])], ignore_index=True)

print("Clean        mean = %.2f, std = %.2f" % (clean.mean(), clean.std()))
print("Contaminated mean = %.2f, std = %.2f" % (contaminated.mean(), contaminated.std()))

One bad value dragged the mean from 11 to 55 and inflated the standard deviation from about 0.8 to 147.

Now ask the Z-score to find it.

In [ ]:
z = (contaminated - contaminated.mean()) / contaminated.std()
print("Z-score of the value 500:", round(z.iloc[-1], 3))
print("Flagged at threshold 3?", abs(z.iloc[-1]) > 3)

**The Z-score fails to flag it.**

The outlier inflated the standard deviation so much that it no longer looks far from the mean when measured in those units. It hid inside the ruler you were using to measure it.

This is called **masking**, and it gets worse with more contamination. With a cluster of extreme values, they collectively stretch the standard deviation until none of them stands out.

There is a second, less well known trap. With `n` data points the largest Z-score that can possibly occur is `(n-1)/sqrt(n)`, no matter how extreme the value is. On small samples that ceiling sits near the thresholds people routinely use.

In [ ]:
for n in [5, 10, 11, 30, 100]:
    print(f"n = {n:>3}  maximum achievable Z-score = {(n - 1) / np.sqrt(n):.3f}")

With 10 observations the maximum is about 2.85, so **a threshold of 3.0 can never flag anything at all**. With 11 it is 3.015, which clears 3.0 by so little that only a value at the absolute theoretical extreme would qualify.

If you apply a fixed Z-score threshold to small groups, for example per customer or per device, some of those groups are mathematically incapable of ever producing an alert. The check reports clean because it cannot do otherwise.

Compare how the robust method handles the same contaminated data.

In [ ]:
print("IQR method flags the 500?", bool(iqr_outliers(contaminated).iloc[-1]))

It catches it without difficulty, because quartiles do not move when you make one value larger.

---
# Part 4: The Modified Z-score

The fix is to keep the Z-score idea but build it from statistics that resist contamination:

- Replace the mean with the **median**.
- Replace the standard deviation with the **MAD**, the Median Absolute Deviation.

The constant 0.6745 rescales MAD so that on normally distributed data the result is comparable to an ordinary Z-score. The usual threshold is 3.5.

In [ ]:
def modified_z_outliers(series, threshold=3.5):
    median = series.median()
    mad = (series - median).abs().median()
    if mad == 0:
        return pd.Series(False, index=series.index)
    modified_z = 0.6745 * (series - median) / mad
    return modified_z.abs() > threshold

In [ ]:
print("Modified Z flags the 500?", bool(modified_z_outliers(contaminated).iloc[-1]))

Caught, where the standard Z-score was blind.

Now apply it to the real dataset.

In [ ]:
print("Height outliers by Modified Z:", modified_z_outliers(people["Height"]).sum())
print("Weight outliers by Modified Z:", modified_z_outliers(people["Weight"]).sum())

See why in a picture before reading the explanation. Plot the Modified Z-scores the same way as the ordinary Z-score above.

In [ ]:
height_median = people["Height"].median()
height_mad = (people["Height"] - height_median).abs().median()
height_mz = 0.6745 * (people["Height"] - height_median) / height_mad

fig, ax = plt.subplots(figsize=(11, 3.5))
plot_scores(height_mz, threshold=3.5, label="Modified Z", ax=ax)
ax.set_xlabel("Row index")
ax.set_title("Height Modified Z-scores, threshold at +/-3.5")
plt.tight_layout()
plt.show()

Zero, where IQR found 8 and the Z-score found 7.

Do not read that as failure. Read it as a **more conservative method disagreeing**, and go and find out why.

The MAD is computed around a median that sits in the valley between the two humps of our mixed population. That makes the typical deviation look large, which widens the threshold, which flags fewer points. The method is behaving exactly as designed. The design assumption, one population, is what does not hold.

The lesson: when methods disagree, the disagreement is information. It is usually telling you something about the data rather than about the code.

## Comparing the three

In [ ]:
comparison = pd.DataFrame({
    "IQR (k=1.5)": [iqr_outliers(people["Height"]).sum(), iqr_outliers(people["Weight"]).sum()],
    "Z-score (3.0)": [z_score_outliers(people["Height"]).sum(), z_score_outliers(people["Weight"]).sum()],
    "Modified Z (3.5)": [modified_z_outliers(people["Height"]).sum(), modified_z_outliers(people["Weight"]).sum()],
}, index=["Height", "Weight"])

comparison

| Method | Uses | Robust to contamination | Assumes normality | Use when |
| :--- | :--- | :--- | :--- | :--- |
| Tukey IQR | Quartiles | Yes | No | Default choice, any distribution |
| Z-score | Mean, std | **No** | Yes | Known clean, roughly normal data |
| Modified Z | Median, MAD | Yes | No | Skewed or heavy tailed data |

If you take one operational rule from this section: **prefer IQR or Modified Z over the plain Z-score for detection work.** The Z-score is the one that fails precisely when there is something to find.

---
# Part 5: Where univariate methods go blind

Everything so far examined one column at a time. That approach has a limit which no amount of threshold tuning can fix.

Plot height against weight.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.scatter(people["Height"], people["Weight"], s=6, alpha=0.25, color="tab:blue")
ax.set_xlabel("Height (inches)")
ax.set_ylabel("Weight (pounds)")
ax.set_title("Height and weight move together")
plt.tight_layout()
plt.show()

The variables are correlated: taller people weigh more. That relationship is the thing univariate methods cannot see.

Consider someone 5 feet 3 inches tall weighing 185 pounds. Check each measurement separately.

In [ ]:
test_height, test_weight = 63.4, 185.2

print(f"Height {test_height} within observed range {people['Height'].min():.1f} to {people['Height'].max():.1f}")
print(f"Weight {test_weight} within observed range {people['Weight'].min():.1f} to {people['Weight'].max():.1f}")
print()
print("Flagged by IQR on height alone?", bool(iqr_outliers(people['Height']).loc[people['Height'].sub(test_height).abs().idxmin()]))
print("Neither measurement is individually unusual.")

Individually, both values are ordinary. **Jointly, the combination is rare**, because that weight is unusual for that height.

Detecting this requires a method that understands the relationship between columns. **Mahalanobis distance** does exactly that. It measures distance from the centre of the data while accounting for how the variables covary, so it effectively asks how far a point is after correcting for the expected relationship.

In [ ]:
features = people[["Height", "Weight"]].to_numpy()
centre = features.mean(axis=0)
covariance = np.cov(features.T)
inv_covariance = np.linalg.inv(covariance)

In [ ]:
deviations = features - centre
mahalanobis = np.sqrt(np.einsum("ij,jk,ik->i", deviations, inv_covariance, deviations))

threshold = np.sqrt(stats.chi2.ppf(0.999, df=2))
maha_outliers = mahalanobis > threshold

print("Mahalanobis outliers:", maha_outliers.sum())

Now compare directly against what the univariate methods found.

In [ ]:
univariate_outliers = (iqr_outliers(people["Height"]) | iqr_outliers(people["Weight"])).to_numpy()

print("Found by univariate IQR         :", univariate_outliers.sum())
print("Found by Mahalanobis            :", maha_outliers.sum())
print("Found ONLY by Mahalanobis       :", (maha_outliers & ~univariate_outliers).sum())

In [ ]:
missed = people[maha_outliers & ~univariate_outliers]
missed[["Gender", "Height", "Weight"]].round(1)

These rows are invisible to every univariate method in this notebook, at any threshold.

Read them and the pattern is clear. Short and heavy, or tall and very light. Each number is individually ordinary. Each **pairing** is not.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.scatter(people["Height"], people["Weight"], s=6, alpha=0.2, color="lightsteelblue", label="Normal")
ax.scatter(people.loc[univariate_outliers, "Height"], people.loc[univariate_outliers, "Weight"],
           s=55, color="tab:orange", label="Caught by univariate IQR", zorder=3)
ax.scatter(missed["Height"], missed["Weight"],
           s=110, facecolors="none", edgecolors="tab:red", linewidths=2,
           label="Only caught jointly", zorder=4)
ax.set_xlabel("Height (inches)")
ax.set_ylabel("Weight (pounds)")
ax.set_title("Univariate methods find the edges, not the interior")
ax.legend()
plt.tight_layout()
plt.show()

This chart is the argument for multivariate detection in one image.

The orange points are at the **outer edges** of the cloud, which is exactly what a per column range check finds. The red circled points sit **inside** the overall range but **off the diagonal**, away from the relationship the population follows.

Real data quality problems very often look like the red points. A transposed pair of fields, a unit conversion applied to one column but not another, or a record stitched together from two different sources all produce values that are individually plausible and jointly impossible.

---
# Part 6: Time series, where the definition of normal moves

There is one more assumption to break. Every method so far assumed rows are **interchangeable**, that shuffling the dataset changes nothing.

For a time series, that is false. Order carries meaning.

Load the NYC taxi dataset, a standard anomaly detection benchmark, recording passenger counts every 30 minutes.

In [ ]:
taxi = pd.read_csv(DATA_DIR / "nyc_taxi.csv", parse_dates=["timestamp"])
print(taxi.shape)
print("From", taxi["timestamp"].min(), "to", taxi["timestamp"].max())
taxi.head()

In [ ]:
daily = taxi.set_index("timestamp").resample("D")["value"].sum()
print("Daily observations:", len(daily))

This benchmark ships with **labelled events**, which is rare and valuable. We know what actually happened on these dates, so we can measure detection honestly rather than admiring plausible looking flags.

In [ ]:
KNOWN_EVENTS = {
    "2014-11-02": "NYC Marathon",
    "2014-11-27": "Thanksgiving",
    "2014-12-25": "Christmas",
    "2015-01-01": "New Year's Day",
    "2015-01-27": "North American Blizzard",
}

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(daily.index, daily.values, color="tab:blue", lw=1)
for date, label in KNOWN_EVENTS.items():
    ts = pd.Timestamp(date)
    if ts in daily.index:
        ax.axvline(ts, color="tab:red", ls="--", alpha=0.7)
        ax.annotate(label, (ts, daily.max()), rotation=90,
                    fontsize=8, va="top", ha="right", color="tab:red")
ax.set_ylabel("Daily passengers")
ax.set_title("NYC taxi ridership with known events marked")
plt.tight_layout()
plt.show()

Some events are visibly dramatic. Others are not. Now apply the statistical methods and score them against the labels.

In [ ]:
daily_iqr = iqr_outliers(daily)
daily_z = z_score_outliers(daily)
daily_mz = modified_z_outliers(daily)

print("Flagged by IQR        :", daily_iqr.sum())
print("Flagged by Z-score    :", daily_z.sum())
print("Flagged by Modified Z :", daily_mz.sum())

In [ ]:
z_values = (daily - daily.mean()) / daily.std()

rows = []
for date, label in KNOWN_EVENTS.items():
    ts = pd.Timestamp(date)
    rows.append({
        "event": label,
        "passengers": int(daily[ts]),
        "z_score": round(z_values[ts], 2),
        "IQR": bool(daily_iqr[ts]),
        "Z-score": bool(daily_z[ts]),
        "Modified Z": bool(daily_mz[ts]),
    })

pd.DataFrame(rows)

---
# Part 7: Read that table carefully

This is the most important result in the activity.

**The blizzard and Christmas were caught by everything.** Ridership collapsed, they are far from the mean in absolute terms, and any method finds them. These are **point anomalies**: extreme values judged against the whole series.

**Thanksgiving was caught only by IQR.** Both Z-score variants missed it. Its Modified Z-score lands around 3.1, just under the conventional 3.5 cutoff, so a threshold chosen by convention rather than by evidence is the entire reason a real holiday went undetected.

**The NYC Marathon and New Year's Day were missed by every single method.**

Look at the Marathon's Z-score: about +0.29. In total daily ridership it is one of the most ordinary days in the entire dataset.

Yet on Marathon day, streets across Manhattan close, the subway reroutes, and taxi movement changes completely. Something enormous happened. The daily total does not show it, because the changes cancel out when you sum over 24 hours.

New Year's Day is the same story. Extremely quiet morning, extremely busy the night before, and the total lands close to average.

## Three kinds of anomaly

This gives us the vocabulary that the rest of the day depends on.

**Point anomaly.** A value extreme against the whole series. The blizzard. Statistical methods find these well, and they are the only kind these methods find reliably.

**Contextual anomaly.** A value that is normal globally but wrong **for its context**. 700,000 passengers is a fine number, unless it is a Sunday in November when the city is closed for a marathon. Detection requires the model to know the context: hour of day, day of week, whether it is a holiday.

**Collective anomaly.** No individual value is unusual, but the **sequence** is. A flat line for six hours where each individual reading is plausible. A pattern that repeats at the wrong frequency. Detection requires looking at windows rather than points.

The taxi results map cleanly onto this. The methods in this notebook caught the point anomalies and were structurally incapable of catching the contextual ones. That is not a tuning problem. No threshold on a daily total will ever surface the Marathon, because the daily total genuinely is normal.

## First attempt at recovering context

The obvious fix is to change the reference population. Instead of comparing each day to the whole series, compare it to **the same weekday**.

In [ ]:
frame = daily.to_frame("passengers")
frame["weekday"] = frame.index.dayofweek
frame["weekday_median"] = frame.groupby("weekday")["passengers"].transform("median")
frame["pct_of_typical"] = frame["passengers"] / frame["weekday_median"]

In [ ]:
for date, label in [("2014-11-02", "NYC Marathon"), ("2015-01-01", "New Year's Day")]:
    ts = pd.Timestamp(date)
    row = frame.loc[ts]
    print(f"{label:16s} {row['passengers']:>8.0f} passengers, "
          f"{row['pct_of_typical']:.1%} of a typical {ts.day_name()}")

**That barely helped.**

The Marathon comes out at about 105 percent of a typical Sunday and New Year's Day at about 91 percent of a typical Thursday. Both are still completely unremarkable. A better reference population did not rescue the detection.

Resist the urge to try a cleverer statistic here. The problem is not the statistic.

## The real culprit is the aggregation

Go back to what we did at the very start of Part 6: we took data recorded **every 30 minutes** and summed it into daily totals.

On Marathon day, ridership is heavily suppressed in some hours and elevated in others. Summing over 24 hours makes those swings **cancel each other out**. The information was not hidden by a weak method. It was destroyed by the aggregation, before any method ran.

No detector can recover a signal that the pipeline already threw away.

Test that claim. Keep the 30 minute granularity, and compare each slot to the median for that same weekday and time of day.

In [ ]:
slots = taxi.set_index("timestamp").copy()
slots["weekday"] = slots.index.dayofweek
slots["time_of_day"] = slots.index.hour * 60 + slots.index.minute

slots["expected"] = slots.groupby(["weekday", "time_of_day"])["value"].transform("median")
slots["residual"] = slots["value"] - slots["expected"]

Now apply the Modified Z-score to the **residuals**, meaning how far each slot sits from what that particular slot normally looks like.

In [ ]:
residual_median = slots["residual"].median()
residual_mad = (slots["residual"] - residual_median).abs().median()
slots["modified_z"] = 0.6745 * (slots["residual"] - residual_median) / residual_mad

slots["is_extreme"] = slots["modified_z"].abs() > 3.5

In [ ]:
per_day = slots.groupby(slots.index.date)["is_extreme"].sum()

print("Extreme 30-minute slots on a typical day (median):", int(per_day.median()))
print()
for date, label in KNOWN_EVENTS.items():
    day = slots.loc[date]
    print(f"{label:26s} extreme slots: {int(day['is_extreme'].sum()):>3}   "
          f"largest |modified Z|: {day['modified_z'].abs().max():.1f}")

Every one of the five events is now clearly detected, including the two that were completely invisible before.

A typical day has **zero** extreme slots. The Marathon has around ten, with a peak Modified Z-score near 15. New Year's Day is even more pronounced. The same simple statistic that failed at daily granularity succeeds easily at 30 minute granularity.

Confirm the detector is finding real events rather than just producing more flags.

In [ ]:
per_day.sort_values(ascending=False).head(10)

Read that list. Blizzard, Independence Day and the day after, the entire Christmas and New Year stretch, Thanksgiving, Labor Day weekend. Almost every top ranked day is a genuine holiday or major event.

## The lesson worth carrying

The most important decision in this notebook was not IQR versus Z-score versus Modified Z. It was **the choice to aggregate to daily totals**, and it was made before any detection code existed.

For a data engineer this is the recurring shape of the problem. Aggregation, sampling rate, partitioning, and retention are all decided early, usually for storage and query cost reasons, and each one silently determines which questions can ever be answered downstream. A detector cannot outperform the granularity it is fed.

When someone reports that anomaly detection is not working, check what the data went through before it arrived. The failure is often upstream of the algorithm.

---
# Your Turn

Work in your own copy under `student-work/week6/day2/`.

## Challenge 1: Detect within groups

Every threshold in Part 2 through Part 4 was computed across a mixed male and female population, which we established is two distributions in a trench coat.

1. Compute IQR outliers for `Height` **separately within each gender**, then combine the results.
2. Compare the total against the 8 found on the pooled data.
3. Identify any row flagged within its group but not in the pooled analysis, and explain in a markdown cell why pooling hid it.

## Challenge 2: Build a threshold sensitivity table

For `Weight`, produce a table with one row per method (IQR, Z-score, Modified Z) and one column per threshold setting, showing the count flagged in each combination. Use at least four threshold values per method.

Then answer: which method's output is most sensitive to its threshold, and what does that imply about deploying it where you cannot easily retune?

## Challenge 3: Hunt the collective anomaly

Point anomalies were covered. Now find a collective one.

1. Resample the taxi data to hourly instead of daily.
2. Compute a 24 hour rolling standard deviation.
3. Find the windows where that rolling standard deviation is unusually **low**, meaning the series went abnormally flat.
4. Plot one of those windows against a normal day.

A flat line in a series that normally has a strong daily rhythm is frequently a **broken sensor or a stalled pipeline**, not a real measurement. Explain why a plain IQR check on the values themselves would never find it.

## Challenge 4: Justify a threshold to a stakeholder

You are adding an automated quality gate to the taxi ingestion pipeline. Flagged days are held for manual review before loading.

Reviewing one day costs approximately 20 minutes of an analyst's time. Loading a genuinely corrupt day means a public dashboard shows wrong numbers until someone notices.

In a markdown cell of no more than 200 words: choose a method and a threshold, estimate the weekly review burden using the counts you measured, and state which of the two errors you chose to favour and why.

There is no correct answer. There is a defensible answer, and that is what you are being asked for.

---
## What you did

- Established that "outlier" is a claim about a reference population, not a property of a row.
- Implemented IQR, Z-score, and Modified Z-score from scratch.
- Demonstrated masking, where a single extreme value hides itself from the Z-score.
- Showed that univariate methods cannot see anomalies that exist only in the relationship between columns.
- Named the three anomaly types: point, contextual, and collective.
- Measured, against labelled real events, that on daily totals every method **missed the NYC Marathon and New Year's Day entirely**.
- Traced that failure to the aggregation rather than to the algorithm, and recovered all five events by detecting at 30 minute granularity against a per weekday, per time of day reference.

## The open problem

Notice what that fix actually required. You had to know, in advance, that the right context was weekday **and** time of day, and you had to build that reference by hand with a `groupby`.

That works because this dataset has one column and one obvious seasonal structure. It does not scale. Given a claims table with forty columns, you cannot hand craft the correct reference population for every combination that might matter, and you will not guess in advance which interactions carry the anomalies.

What you want is a method that **learns the structure itself**, across many columns at once, without being told where to look.

Next: **Activity 3**, where Isolation Forest does exactly that, and where you find out what it costs you in explainability to get it.